In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('/kaggle/input/competitions/aptos2019-blindness-detection/train.csv')

df['file_path'] = df['id_code'].apply(
    lambda x: f'/kaggle/input/competitions/aptos2019-blindness-detection/train_images/{x}.png'
)

print(df['diagnosis'].value_counts())

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['diagnosis'],
    random_state=42
)

print(f"Train: {len(train_df)}, Val: {len(val_df)}")

diagnosis
0    1805
2     999
1     370
4     295
3     193
Name: count, dtype: int64
Train: 2929, Val: 733


In [2]:
import tensorflow as tf
import cv2
import numpy as np

IMG_SIZE = 224

def preprocess_fundus_tf(file_path, label):
    def _process(path):
        path = path.numpy().decode('utf-8')
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
        coords = cv2.findNonZero(mask)
        x, y, w, h = cv2.boundingRect(coords)
        cropped = img[y:y+h, x:x+w]
        h, w = cropped.shape[:2]
        size = max(h, w)
        pad_top = (size - h) // 2
        pad_bottom = size - h - pad_top
        pad_left = (size - w) // 2
        pad_right = size - w - pad_left
        padded = cv2.copyMakeBorder(cropped, pad_top, pad_bottom, pad_left, pad_right,
                                      cv2.BORDER_CONSTANT, value=[0, 0, 0])
        resized = cv2.resize(padded, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        return resized.astype(np.float32)  # NO /255 — EfficientNet handles this internally

    img = tf.py_function(_process, [file_path], tf.float32)
    img.set_shape((IMG_SIZE, IMG_SIZE, 3))
    return img, label

def make_dataset(df, batch_size=32, shuffle=True):
    paths = df['file_path'].values
    labels = df['diagnosis'].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df))
    ds = ds.map(preprocess_fundus_tf, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)

I0000 00:00:1789075238.212840      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [3]:
from tensorflow.keras import layers, models

base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(224, 224, 3), include_top=False, weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(5, activation='softmax')
])

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
import tensorflow as tf

def distance_weighted_loss(y_true, y_pred, num_classes=5, distance_weight=0.5):
    y_true = tf.cast(y_true, tf.int32)
    cce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)

    class_indices = tf.range(num_classes, dtype=tf.float32)
    true_class_f = tf.cast(y_true, tf.float32)
    distances = tf.abs(tf.expand_dims(true_class_f, axis=1) - class_indices)
    expected_distance = tf.reduce_sum(y_pred * distances, axis=1)

    return cce + distance_weight * expected_distance

In [5]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(train_df['diagnosis'])
weights = compute_class_weight('balanced', classes=classes, y=train_df['diagnosis'])
class_weights = dict(zip(classes, weights))

In [6]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=distance_weighted_loss,
    metrics=['accuracy']
)

In [7]:
history_ordinal = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights
)

Epoch 1/10


I0000 00:00:1789075278.515704      66 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


92/92 ━━━━━━━━━━━━━━━━━━━━ 276s 3s/step - accuracy: 0.6122 - loss: 1.7382 - val_accuracy: 0.6889 - val_loss: 1.0576
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 214s 2s/step - accuracy: 0.6835 - loss: 1.3912 - val_accuracy: 0.7312 - val_loss: 0.9613
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 212s 2s/step - accuracy: 0.7095 - loss: 1.3077 - val_accuracy: 0.7121 - val_loss: 0.9684
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 210s 2s/step - accuracy: 0.7272 - loss: 1.2433 - val_accuracy: 0.6917 - val_loss: 1.0523
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 274s 2s/step - accuracy: 0.7364 - loss: 1.2040 - val_accuracy: 0.7231 - val_loss: 0.9268
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 220s 2s/step - accuracy: 0.7334 - loss: 1.2071 - val_accuracy: 0.7462 - val_loss: 0.8608
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 221s 2s/step - accuracy: 0.7388 - loss: 1.1654 - val_accuracy: 0.7271 - val_loss: 0.9123
Epoch 8/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 222s 2s/step - accuracy: 0.7511 - loss: 1.1405 - val_accuracy: 0.7640 - val_loss: 0.

In [8]:
from sklearn.metrics import confusion_matrix, classification_report

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.95      0.95       361
           1       0.51      0.57      0.54        74
           2       0.75      0.58      0.65       200
           3       0.26      0.54      0.35        39
           4       0.49      0.47      0.48        59

    accuracy                           0.75       733
   macro avg       0.59      0.62      0.60       733
weighted avg       0.78      0.75      0.76       733

[[343  14   4   0   0]
 [ 10  42  18   1   3]
 [  5  19 116  43  17]
 [  0   1   8  21   9]
 [  0   6   9  16  28]]


In [9]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weights
)

Epoch 1/5


2026-09-10 22:01:29.265387: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-10 22:01:29.468567: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-10 22:01:30.089793: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-10 22:01:30.295863: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


91/92 ━━━━━━━━━━━━━━━━━━━━ 1s 2s/step - accuracy: 0.5843 - loss: 1.2499

2026-09-10 22:04:25.869190: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-10 22:04:26.077396: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-10 22:04:26.467900: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-10 22:04:26.676370: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


92/92 ━━━━━━━━━━━━━━━━━━━━ 268s 2s/step - accuracy: 0.6115 - loss: 1.1686 - val_accuracy: 0.6944 - val_loss: 0.7563
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 212s 2s/step - accuracy: 0.6903 - loss: 0.9715 - val_accuracy: 0.6862 - val_loss: 0.7582
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 215s 2s/step - accuracy: 0.7269 - loss: 0.9100 - val_accuracy: 0.6808 - val_loss: 0.7538
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 266s 2s/step - accuracy: 0.7330 - loss: 0.8711 - val_accuracy: 0.6903 - val_loss: 0.7431
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 218s 2s/step - accuracy: 0.7378 - loss: 0.8668 - val_accuracy: 0.6876 - val_loss: 0.7387


In [10]:
model.save('/kaggle/working/dr_model_ordinal.keras')
